In [3]:
####IMPORTING
import re
import numpy as np

with open("data_6/inputs_6.txt", "r") as f:
    text = f.read()

blocks = re.findall(r'\[.*?\](?=\s*\[|$)', text, re.DOTALL)
inputs_raw = [eval(b, {"array": np.array}) for b in blocks]


with open("data_6/outputs_6.txt", "r") as f:
    text = f.read()

blocks = re.findall(r'\[.*?\](?=\s*\[|$)', text, re.DOTALL)
outputs_raw = [eval(b, {"np": np}) for b in blocks]

print(inputs_raw)
print(outputs_raw)

[[array([0.67543 , 0.748945]), array([0.058084, 0.866176]), array([0.463538, 0.585335, 0.311607]), array([0.531561, 0.342526, 0.397711, 0.198366]), array([0.366362, 0.45607 , 0.785176, 0.199674]), array([0.743898, 0.127473, 0.738098, 0.636447, 0.075186]), array([0.948886, 0.965632, 0.808397, 0.304614, 0.097672, 0.684233]), array([0.055773, 0.01307 , 0.064056, 0.      , 0.414379, 0.703072,
       0.421898, 0.902928])], [array([0.080919, 0.40298 ]), array([0.879273, 0.779203]), array([0.59192 , 0.7255  , 0.079854]), array([0.416983, 0.452178, 0.388383, 0.310864]), array([0.158875, 0.808215, 0.943956, 0.901422]), array([0.48091 , 0.425612, 0.348942, 0.809975, 0.255399]), array([0.580779, 0.526972, 0.351037, 0.493213, 0.365097, 0.657931]), array([0.122085, 0.080745, 0.011016, 0.224789, 0.742507, 0.480219,
       0.16568 , 0.248194])], [array([0.411765, 0.096892]), array([0.738382, 0.987009]), array([0.531437, 0.743237, 0.013497]), array([0.396863, 0.455675, 0.411971, 0.235379]), array([0.3

In [5]:
# ======================
# RESTRUCTURE DATA
# ======================

n_iterations = len(inputs_raw)   # 5
n_functions = len(inputs_raw[0]) # 8

inputs_15 = []
outputs_15 = []

for f in range(n_functions):
    X_f = []
    y_f = []

    for t in range(n_iterations):
        X_f.append(inputs_raw[t][f])
        y_f.append(outputs_raw[t][f])

    X_f = np.vstack(X_f)
    y_f = np.array(y_f, dtype=float)

    inputs_15.append(X_f)
    outputs_15.append(y_f)

# check
for i in range(8):
    print(f"Function {i+1}: {inputs_15[i].shape}, {outputs_15[i].shape}")

print(outputs_15)

Function 1: (5, 2), (5,)
Function 2: (5, 2), (5,)
Function 3: (5, 3), (5,)
Function 4: (5, 4), (5,)
Function 5: (5, 4), (5,)
Function 6: (5, 5), (5,)
Function 7: (5, 6), (5,)
Function 8: (5, 8), (5,)
[array([-1.55658039e-013,  5.50990631e-082,  4.58701125e-074,
       -1.25368270e-230, -7.63689547e-170]), array([0.09550182, 0.37388089, 0.46092492, 0.0664094 , 0.37505844]), array([-0.06028347, -0.04869721, -0.07946811, -0.04404363, -0.1242321 ]), array([ -4.37440006,  -1.00059496,  -2.50043244, -22.06877914,
        -7.55541593]), array([4.42227225e-01, 1.40292112e+03, 1.38504500e+03, 2.07235582e+01,
       2.19315337e+03]), array([-0.83555188, -0.59597613, -0.53415927, -1.87727621, -0.55452449]), array([3.45079114e-02, 7.94642820e-01, 1.11153414e+00, 7.25544558e-04,
       1.41496775e+00]), array([9.71854829, 9.9293868 , 9.49864081, 9.07168405, 9.90221386])]


In [9]:
# ======================
# MODEL
# ======================
def choose_next_point_v6(X, y):

    dim = X.shape[1]

    # normalize y
    y_norm = (y - y.mean()) / (y.std() + 1e-8)

    # SMALLER model (important!)
    def small_model():
        model = tf.keras.Sequential([
            tf.keras.Input(shape=(dim,)),
            tf.keras.layers.Dense(16, activation='relu'),
            tf.keras.layers.Dense(16, activation='relu'),
            tf.keras.layers.Dense(1)
        ])
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
            loss='mse'
        )
        return model

    # train small ensemble
    models = []
    for _ in range(3):
        m = small_model()
        m.fit(X, y_norm, epochs=80, verbose=0)
        models.append(m)

    best_x = X[np.argmax(y)]

    candidates = []

    # (A) local refinement (MAIN)
    for _ in range(6):
        noise = np.random.normal(0, 0.05, dim)
        x_new = np.clip(best_x + noise, 0, 1)
        candidates.append(x_new)

    # (B) few gradient steps (light use!)
    for m in models:
        x = tf.Variable(best_x.reshape(1, -1), dtype=tf.float32)

        for _ in range(15):  # fewer steps
            with tf.GradientTape() as tape:
                tape.watch(x)
                pred = m(x)

            grads = tape.gradient(pred, x)
            grads = tf.clip_by_value(grads, -0.05, 0.05)

            x.assign_add(0.003 * grads)
            x.assign(tf.clip_by_value(x, 0, 1))

        candidates.append(x.numpy().flatten())

    # (C) small exploration
    for _ in range(2):
        candidates.append(np.random.uniform(0, 1, dim))

    # scoring
    best_score = -np.inf
    best_candidate = None

    for x in candidates:
        preds = np.array([m.predict(x.reshape(1, -1), verbose=0)[0,0] for m in models])
        mu = preds.mean()
        sigma = preds.std()

        score = mu + 0.1 * sigma

        if score > best_score:
            best_score = score
            best_candidate = x

    return best_candidate

In [10]:
# ======================
# PRINT FINAL QUERIES
# ======================
queries = []

for i in range(8):
    X = inputs_15[i]
    y = outputs_15[i]

    q = choose_next_point_v6(X, y)
    queries.append(q)

for i, q in enumerate(queries):
    print(f"Function {i+1}: " + "-".join([f"{x:.6f}" for x in q]))

Function 1: 0.304242-0.524756
Function 2: 0.811664-0.975720
Function 3: 0.828738-0.356753-0.280935
Function 4: 0.314356-0.508571-0.907566-0.249292
Function 5: 0.101739-0.808101-0.975892-0.961625
Function 6: 0.504890-0.335599-0.373531-0.836990-0.208024
Function 7: 0.031731-0.493032-0.244212-0.250795-0.367196-0.906977
Function 8: 0.025351-0.962648-0.835980-0.695974-0.408953-0.173294-0.156437-0.250243
